# L-1 Step 2：Embedding + KNN 分類

對 Step 1 未覆蓋（MEDIUM / SKIP）的日報，使用 Embedding 向量相似度 + KNN 自動分類 L1–L7。

| 子任務 | 規格 |
|--------|------|
| A 讀取 Step 1 MEDIUM/SKIP | 從 step1_results.jsonl 篩出待處理筆數 |
| B 向量化 | gemini-embedding-2-preview，格式「[L層][DayN] 文字」，批次 100 |
| C KNN 查詢 | Vertex AI Vector Search，k=10，信心=Top-1 相似度 |
| D 信心路由 | >=0.65->直接定案；0.50-0.65->進 Step 3；<0.50->uncertain |
| E 信心分布報告 | 各區間比例 + 各層覆蓋數驗證（目標每層 >= 3 萬篇） |

| 完成標準 | 目標 |
|----------|------|
| KNN 覆蓋率（HIGH >= 0.65） | 38–48% |
| 每 L 層有效標注 | >= 3 萬篇 |
| uncertain 比例 | < 20% |

> 向量庫：Step 0 日報種子 2,375 筆（INDEX_ID = 150633093005312000）


## 0. 安裝套件

In [ ]:
!pip install google-genai google-cloud-aiplatform pyodbc pandas pyarrow tqdm -q

## 1. 匯入套件 & 全域設定

In [8]:
import os, re, json, time, hashlib, configparser
from pathlib import Path
from datetime import datetime

import pyodbc
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types as genai_types

# ── 路徑設定 ──────────────────────────────────────────
CFG_GCP    = r'D:\yujui\GoogleCloud.ini'
CFG_SQL    = r'D:\yujui\SqlServer18.txt'

STEP1_DIR  = Path(r'D:\yujui\痛點需求地圖\step1_output')
OUTPUT_DIR = Path(r'D:\yujui\痛點需求地圖\step2_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEP1_JSONL  = STEP1_DIR  / 'step1_results.jsonl'
RESULT_JSONL = OUTPUT_DIR / 'step2_results.jsonl'
COVERAGE_CSV = OUTPUT_DIR / 'step2_coverage.csv'

# ── Vertex AI Vector Search 設定 ──────────────────────
INDEX_ID         = '150633093005312000'
INDEX_RESOURCE   = f'projects/648642612568/locations/asia-east1/indexes/{INDEX_ID}'
REGION           = 'asia-east1'
EMBED_MODEL      = 'gemini-embedding-2-preview'
EMBED_BATCH_SIZE = 100

# ── 信心門檻 ──────────────────────────────────────────
HIGH_THRESH   = 0.65   # >= HIGH_THRESH -> 直接定案
MEDIUM_THRESH = 0.50   # MEDIUM_THRESH ~ HIGH_THRESH -> 進 Step 3
                       # < MEDIUM_THRESH -> uncertain

KNN_K        = 10
SQL_DATABASE = 'DSC_CRM_SP'

print(f'輸出目錄：{OUTPUT_DIR}')
print(f'Index   ：{INDEX_RESOURCE}')


輸出目錄：D:\yujui\痛點需求地圖\step2_output
Index   ：projects/648642612568/locations/asia-east1/indexes/150633093005312000


## 2. 載入設定 & 建立連線

In [9]:
# ── GCP 設定 ──────────────────────────────────────────
cfg = configparser.ConfigParser()
cfg.read(CFG_GCP, encoding='utf-8')
g = cfg['gcp']

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = g['service_account_key']
PROJECT_ID     = g['project_id']
GEMINI_API_KEY = g['gemini_api_key']

client = genai.Client(api_key=GEMINI_API_KEY)
print(f'GCP Project：{PROJECT_ID}')

# ── SQL Server 連線 ───────────────────────────────────
cfg_sql = configparser.ConfigParser()
cfg_sql.read(CFG_SQL, encoding='utf-8')
if 'mssql' not in cfg_sql:
    raise RuntimeError(f'找不到 [mssql] 區段：{CFG_SQL}')
sec = cfg_sql['mssql']
conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};SERVER={sec['server']};DATABASE={SQL_DATABASE};"
    f"UID={sec['uid']};PWD={sec['pwd']};",
    autocommit=True
)
print(f'SQL Server 連線成功：{sec["server"]}')


GCP Project：digiwin-ai
SQL Server 連線成功：10.20.99.18


## 3. 初始化 Vertex AI Vector Search

In [10]:
import google.cloud.aiplatform as aip
import warnings
warnings.filterwarnings('ignore')

aip.init(project=PROJECT_ID, location=REGION)
index = aip.MatchingEngineIndex(INDEX_RESOURCE)
print(f'Index 載入成功：{index.display_name}')
print(f'  resource name: {index.resource_name}')


Index 載入成功：l1l7-seed-index
  resource name: projects/648642612568/locations/asia-east1/indexes/150633093005312000


---
## 子任務 A：讀取 Step 1 MEDIUM / SKIP 筆數
只處理 Step 1 未定案的筆數（信心 MEDIUM 或 SKIP）。


In [ ]:
# 只統計各信心等級數量，不建 DataFrame（節省 kernel 記憶體）
stats_all = {'HIGH': 0, 'MEDIUM': 0, 'SKIP': 0}
with open(STEP1_JSONL, encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        conf = json.loads(line).get('step1_confidence', '')
        stats_all[conf] = stats_all.get(conf, 0) + 1

total_step1 = sum(stats_all.values())
print(f'Step 1 總筆數    ：{total_step1:,}')
print(f'HIGH（已定案）  ：{stats_all["HIGH"]:,}  ({stats_all["HIGH"]/total_step1*100:.1f}%)')
print(f'MEDIUM（待 KNN）：{stats_all["MEDIUM"]:,}  ({stats_all["MEDIUM"]/total_step1*100:.1f}%)')
print(f'SKIP  （短文本）：{stats_all["SKIP"]:,}  ({stats_all["SKIP"]/total_step1*100:.1f}%)')


In [ ]:
import gc

TEMP_MATCHED = OUTPUT_DIR / '_a2_matched.jsonl'

# ── 若 _a2_matched.jsonl 已存在，直接讀（跳過 SQL scan）──
if TEMP_MATCHED.exists():
    todo_df = pd.read_json(TEMP_MATCHED, lines=True)
    todo_df['log_len'] = pd.to_numeric(todo_df['log_len'], errors='coerce').fillna(0).astype(int)
    print(f'從已有檔案載入：{len(todo_df):,} 筆（{TEMP_MATCHED}）')
    print(todo_df.dtypes)

# ── 若不存在，執行 subprocess SQL scan（約 10–20 分鐘）────
else:
    import subprocess, sys
    SCAN_SCRIPT = OUTPUT_DIR / '_scan_a2.py'

    SCAN_SCRIPT.write_text(f'''
import json, hashlib, configparser, pyodbc
from pathlib import Path

STEP1_JSONL  = Path(r"{STEP1_JSONL}")
TEMP_MATCHED = Path(r"{TEMP_MATCHED}")
CFG_SQL      = r"{CFG_SQL}"
SQL_DATABASE = "{SQL_DATABASE}"

todo_ids = set()
with open(STEP1_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        r = json.loads(line)
        if r.get("step1_confidence") == "MEDIUM":
            todo_ids.add(r["event_id"])

total_need = len(todo_ids)
print(f"todo_ids: {{total_need:,}}", flush=True)

cfg = configparser.ConfigParser()
cfg.read(CFG_SQL, encoding="utf-8")
sec = cfg["mssql"]
conn = pyodbc.connect(
    f"DRIVER={{{{SQL Server}}}};SERVER={{sec[\'server\']}};DATABASE={{SQL_DATABASE}};"
    f"UID={{sec[\'uid\']}};PWD={{sec[\'pwd\']}};",
    autocommit=True
)
query = """
SELECT GY001, GY012, LEN(GY012), FORMAT(CREATE_DATE,'yyyy-MM')
FROM CRMGY
WHERE LEN(LTRIM(RTRIM(GY012))) >= 5
  AND GY012 NOT LIKE N\'%MA\u7c3d\u56de%\'
  AND GY012 NOT LIKE N\'%MA\u7c3d\u7d04%\'
  AND GY012 NOT LIKE N\'%\u7dad\u8b77\u5408\u7d04%\'
  AND GY012 NOT LIKE N\'%\u5ee2\u6b62%\'
"""
cursor = conn.cursor()
cursor.execute(query)
found_count = 0
batch_no    = 0
with open(TEMP_MATCHED, "w", encoding="utf-8") as fout:
    while True:
        rows = cursor.fetchmany(10000)
        if not rows: break
        for row in rows:
            lc  = str(row[1]).replace("\\n", " ")
            eid = hashlib.sha256(lc.encode()).hexdigest()[:16]
            if eid in todo_ids:
                todo_ids.discard(eid)
                fout.write(json.dumps({{"event_id":eid,"log_content":lc,
                    "log_len":int(row[2]) if row[2] else 0,
                    "ym":str(row[3]) if row[3] else "",
                    "company_id":str(row[0]) if row[0] else ""}},
                    ensure_ascii=False) + "\\n")
                found_count += 1
        del rows
        batch_no += 1
        if batch_no % 50 == 0:
            print(f"  batch {{batch_no:>4}}: {{found_count:,}}/{{total_need:,}}", flush=True)
        if not todo_ids:
            print(f"  全部找到，提前結束 batch {{batch_no}}", flush=True)
            break
cursor.close()
conn.close()
print(f"完成：{{found_count:,}} 筆", flush=True)
''', encoding='utf-8')

    print('開始 SQL scan（獨立進程，約 10–20 分鐘）...')
    result = subprocess.run(
        [sys.executable, str(SCAN_SCRIPT)],
        capture_output=True, text=True, timeout=7200,
        env={**__import__('os').environ}
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'scan script 失敗（return code {result.returncode}）')

    todo_df = pd.read_json(TEMP_MATCHED, lines=True)
    todo_df['log_len'] = pd.to_numeric(todo_df['log_len'], errors='coerce').fillna(0).astype(int)
    gc.collect()

print(f'\n最終待處理：{len(todo_df):,} 筆')
if len(todo_df) == 0:
    raise RuntimeError('todo_df 為空')


---
## 子任務 B：向量化
格式：`[L步驟][DayN] 文字`，使用 `gemini-embedding-2-preview`，批次 100 筆。


In [ ]:
def fmt_embed_text(row: dict) -> str:
    day  = min(int(row.get('log_len', 100)) // 5, 999)
    text = str(row.get('log_content', ''))[:300]
    return f'[STEP2][Day{day}] {text}'

def embed_batch(texts: list) -> list:
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=genai_types.EmbedContentConfig(task_type='RETRIEVAL_QUERY'),
    )
    return [list(e.values) for e in result.embeddings]

# ── Seed ID → Layer 對照表 ─────────────────────────
SEED_LAYER = {}
with open(
    r"D:\yujui\痛點需求地圖\seed_output\labeled_logs.jsonl",
    encoding="utf-8"
) as _f:
    for _l in _f:
        _s = json.loads(_l)
        SEED_LAYER[_s["event_id"]] = _s.get("l_layer", "uncertain")
print(f"Seed 對照表：{len(SEED_LAYER):,} 筆")

def parse_neighbors(neighbors: list) -> tuple:
    """KNN 近鄰 → (top1_sim, top1_id, top1_layer)"""
    if not neighbors:
        return 0.0, '', 'uncertain'
    top1     = neighbors[0]
    top1_sim = round(1.0 - float(top1.distance), 4)
    top1_id  = top1.id
    # 優先從 seed 對照表查 layer，fallback 才用 restricts
    top1_layer = SEED_LAYER.get(top1_id)
    if not top1_layer:
        top1_layer = (
            top1.restricts[0].allow_tokens[0]
            if hasattr(top1, 'restricts') and top1.restricts
            else 'uncertain'
        )
    return top1_sim, top1_id, top1_layer

# ── smoke test ────────────────────────────────────────
if len(todo_df) == 0:
    raise RuntimeError('todo_df 為空，請確認 Cell A2 已正確執行')

_sample = todo_df.iloc[0].to_dict()
_vec    = embed_batch([fmt_embed_text(_sample)])[0]
print(f'向量維度：{len(_vec)}，型別：{type(_vec[0])}')
print(f'todo_df ：{len(todo_df):,} 筆')

In [ ]:
# ── B+C 串流 Pipeline：Embed → KNN（批次）→ 立刻寫入 + 可選存向量
# 前置：Cell C2 必須先執行（建立/載入 index_endpoint + DEPLOYED_INDEX_ID）
#
# SAVE_VECTORS = True  → 額外存 parquet（~3-5 GB）；未來種子庫更新可免費重跑 KNN
# SAVE_VECTORS = False → 不存向量（省空間，種子庫確定不動時用）
#
# FORCE_RESTART = True  → 刪除舊結果重跑（embed_batch 修正後第一次執行用）
# FORCE_RESTART = False → 斷點續跑（中途中斷時改為 False 接續）

import pyarrow as pa
import pyarrow.parquet as pq
import os

SAVE_VECTORS   = True
VECTOR_PARQUET = OUTPUT_DIR / 'step2_vectors.parquet'
RESUME         = True
FORCE_RESTART  = True   # 第一次跑完後改 False 即可斷點續跑
EMBED_SLEEP    = 0.1

# ── 強制重跑：刪除舊錯誤結果 ──────────────────────────
if FORCE_RESTART:
    if RESULT_JSONL.exists():
        os.remove(RESULT_JSONL)
        print(f"已刪除舊結果：{RESULT_JSONL}")
    if SAVE_VECTORS and VECTOR_PARQUET.exists():
        os.remove(VECTOR_PARQUET)
        print(f"已刪除舊向量：{VECTOR_PARQUET}")

# ── 斷點續跑 ──────────────────────────────────────────
done_ids = set()
if RESUME and not FORCE_RESTART and RESULT_JSONL.exists():
    with open(RESULT_JSONL, encoding="utf-8") as f:
        done_ids = {json.loads(l)["event_id"] for l in f if l.strip()}
    print(f"斷點續跑：已完成 {len(done_ids):,} 筆")

embed_todo = todo_df[~todo_df["event_id"].isin(done_ids)].reset_index(drop=True)
total = len(embed_todo)
print(f"待處理：{total:,} 筆  |  預估批次：{(total-1)//EMBED_BATCH_SIZE+1:,}")
if SAVE_VECTORS:
    print(f"向量存檔：{VECTOR_PARQUET}")

if total == 0:
    print("全部已完成，跳過。")
else:
    _vec_schema = pa.schema([
        ("event_id", pa.string()),
        ("embedding", pa.list_(pa.float32())),
    ])
    _pq_writer = pq.ParquetWriter(str(VECTOR_PARQUET), _vec_schema) if SAVE_VECTORS else None

    written = 0
    errors  = 0

    with open(RESULT_JSONL, "a", encoding="utf-8") as fout:
        for i in tqdm(range(0, total, EMBED_BATCH_SIZE), desc="Embed→KNN→Write"):
            batch = embed_todo.iloc[i : i + EMBED_BATCH_SIZE]
            rows  = batch.to_dict("records")
            texts = [fmt_embed_text(r) for r in rows]

            # Embed 100 筆（1 次 API）
            try:
                vecs = embed_batch(texts)
            except Exception as e:
                print(f"[Embed 錯誤] batch={i}: {e}")
                errors += len(rows)
                time.sleep(2)
                continue

            # 存向量到 parquet（串流，不佔記憶體）
            if _pq_writer:
                _pq_writer.write_table(pa.table({
                    "event_id":  [r["event_id"] for r in rows],
                    "embedding": [list(map(float, v)) for v in vecs],
                }, schema=_vec_schema))

            # KNN 100 筆（1 次 API，比逐筆快 100x）
            try:
                all_neighbors = index_endpoint.find_neighbors(
                    deployed_index_id=DEPLOYED_INDEX_ID,
                    queries=vecs,
                    num_neighbors=KNN_K,
                )
            except Exception as e:
                print(f"[KNN 錯誤] batch={i}: {e}")
                all_neighbors = [[] for _ in vecs]
                errors += 1

            # 立刻寫入結果
            for row, neighbors in zip(rows, all_neighbors):
                top1_sim, top1_id, top1_layer = parse_neighbors(neighbors)
                if top1_sim >= HIGH_THRESH:
                    status = "HIGH"
                elif top1_sim >= MEDIUM_THRESH:
                    status = "MEDIUM"
                else:
                    status, top1_layer = "LOW", "uncertain"

                fout.write(json.dumps({
                    "event_id":         row["event_id"],
                    "company_id":       row.get("company_id", ""),
                    "ym":               row.get("ym", ""),
                    "step2_result":     top1_layer,
                    "step2_confidence": top1_sim,
                    "step2_status":     status,
                    "top1_seed_id":     top1_id,
                    "classified_at":    datetime.now().isoformat(),
                }, ensure_ascii=False) + "
")
                written += 1

            time.sleep(EMBED_SLEEP)

    if _pq_writer:
        _pq_writer.close()
        size_gb = os.path.getsize(VECTOR_PARQUET) / 1e9
        print(f"向量存檔完成：{VECTOR_PARQUET}  ({size_gb:.2f} GB)")

    print(f"分類完成：{written:,} 筆 → {RESULT_JSONL}")
    if errors:
        print(f"錯誤批次：{errors} 個（已跳過）")


---
## 子任務 C：KNN 查詢（Vertex AI Vector Search）
對每筆向量查詢 Index，取 Top-k 近鄰，信心 = Top-1 相似度。


In [ ]:
# ── C1：KNN 單筆診斷測試 ───────────────────────────────────
# B2 跑完全 LOW 時先執行此 cell，確認 find_neighbors 是否正常

sample     = todo_df.iloc[0].to_dict()
test_text  = fmt_embed_text(sample)
test_vec   = embed_batch([test_text])[0]
test_vec_l = list(test_vec)   # 確保是 Python list（非 protobuf object）

print(f'向量維度：{len(test_vec_l)}，型別：{type(test_vec_l[0])}')
print(f'測試 event_id：{sample["event_id"]}')

try:
    result = index_endpoint.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[test_vec_l],
        num_neighbors=KNN_K,
    )
    neighbors = result[0]
    print(f'\n✅ find_neighbors 成功，回傳 {len(neighbors)} 個近鄰')
    for i, n in enumerate(neighbors[:3]):
        sim = round(1.0 - float(n.distance), 4)
        print(f'  [{i}] id={n.id}  similarity={sim}')
except Exception as e:
    print(f'\n❌ find_neighbors 失敗：{type(e).__name__}: {e}')


In [12]:
# ── C2：建立 / 載入 Index Endpoint（一次性，約 10–20 分鐘）───
# 機器規格對應：
#   SHARD_SIZE_SMALL  → e2-standard-2 / e2-standard-16
#   SHARD_SIZE_MEDIUM → e2-standard-16 / e2-highmem-16   ← 本 Index 適用
#   SHARD_SIZE_LARGE  → e2-highmem-16 / n1-standard-16

CREATE_ENDPOINT   = False  # 已建立，直接載入
ENDPOINT_RESOURCE = 'projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736'
DEPLOYED_INDEX_ID = 'l1l7_seed_deployed'

if CREATE_ENDPOINT:
    print('建立 IndexEndpoint 中...')
    index_endpoint = aip.MatchingEngineIndexEndpoint.create(
        display_name='l1l7-seed-endpoint',
        public_endpoint_enabled=True,
    )
    print(f'Endpoint 建立完成：{index_endpoint.resource_name}')

    print('部署 Index（約 10–20 分鐘）...')
    index_endpoint.deploy_index(
        index=index,
        deployed_index_id=DEPLOYED_INDEX_ID,
        display_name='l1l7-seed-deployed',
        machine_type='e2-standard-16',
        min_replica_count=1,
        max_replica_count=1,
    )
    ENDPOINT_RESOURCE = index_endpoint.resource_name
    print(f'ENDPOINT_RESOURCE = "{ENDPOINT_RESOURCE}"')
else:
    index_endpoint = aip.MatchingEngineIndexEndpoint(ENDPOINT_RESOURCE)
    print(f'Endpoint 載入：{index_endpoint.display_name}')
    print(f'  resource: {index_endpoint.resource_name}')


Endpoint 載入：l1l7-seed-endpoint
  resource: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736


In [ ]:
# ── C3 已廢棄 ──────────────────────────────────────────────
# Embed + KNN + Write 已合併至 Cell B2（串流 pipeline）
# 請直接執行 Cell B2，勿執行本 Cell
print('Cell C3 已廢棄，請執行 Cell B2（Embed→KNN→Write 串流 pipeline）')


---
## 子任務 E：信心分布報告
驗證 HIGH 覆蓋率 38–48%，各層覆蓋數，以及進入 Step 3 的比例。


In [5]:
results2 = [json.loads(l) for l in open(RESULT_JSONL, encoding='utf-8') if l.strip()]
df2 = pd.DataFrame(results2)

total2  = len(df2)
high_n  = (df2['step2_status'] == 'HIGH').sum()
med_n   = (df2['step2_status'] == 'MEDIUM').sum()
low_n   = (df2['step2_status'] == 'LOW').sum()

print(f'Step 2 總筆數           ：{total2:,}')
print(f'HIGH  (>=0.65，直接定案)：{high_n:,}  {high_n/total2*100:.1f}%')
print(f'MEDIUM (0.50-0.65，進 Step 3)：{med_n:,}  {med_n/total2*100:.1f}%')
print(f'LOW   (<0.50，uncertain)：{low_n:,}  {low_n/total2*100:.1f}%')

# 各層分布（HIGH only）
high_df2   = df2[df2['step2_status'] == 'HIGH']
layer_dist = high_df2['step2_result'].value_counts().sort_index()
print('\n【HIGH 各層分布】')
for layer in ['L1','L2','L3','L4','L5','L6','L7','uncertain']:
    n = int(layer_dist.get(layer, 0))
    print(f'  {layer}：{n:>8,} 筆')


Step 2 總筆數           ：935,570
HIGH  (>=0.65，直接定案)：935,556  100.0%
MEDIUM (0.50-0.65，進 Step 3)：14  0.0%
LOW   (<0.50，uncertain)：0  0.0%

【HIGH 各層分布】
  L1： 343,184 筆
  L2： 176,434 筆
  L3： 114,686 筆
  L4：  25,695 筆
  L5： 119,093 筆
  L6： 107,900 筆
  L7：  48,528 筆
  uncertain：      36 筆


In [6]:
report_rows = []
for layer in ['L1','L2','L3','L4','L5','L6','L7','uncertain']:
    n = int(layer_dist.get(layer, 0))
    report_rows.append({'l_layer': layer, 'high_count': n,
                        'pass_30k': n >= 30000})
report_rows.append({'l_layer': 'TOTAL_HIGH',   'high_count': int(high_n), 'pass_30k': high_n/total2 >= 0.38})
report_rows.append({'l_layer': 'TOTAL_MEDIUM', 'high_count': int(med_n),  'pass_30k': True})
report_rows.append({'l_layer': 'TOTAL_LOW',    'high_count': int(low_n),  'pass_30k': True})

pd.DataFrame(report_rows).to_csv(COVERAGE_CSV, index=False, encoding='utf-8-sig')

print('=' * 55)
print('Step 2 完成')
print(f'  step2_results.jsonl : {RESULT_JSONL}')
print(f'  step2_coverage.csv  : {COVERAGE_CSV}')
coverage2 = high_n / total2 * 100
status = '✅' if coverage2 >= 38 else '❌ 覆蓋率不足'
print(f'  HIGH 覆蓋率：{coverage2:.1f}%  {status}')
print(f'  -> Step 3 待處理：{med_n:,} 筆（信心 0.50-0.65）')


Step 2 完成
  step2_results.jsonl : D:\yujui\痛點需求地圖\step2_output\step2_results.jsonl
  step2_coverage.csv  : D:\yujui\痛點需求地圖\step2_output\step2_coverage.csv
  HIGH 覆蓋率：100.0%  ✅
  -> Step 3 待處理：14 筆（信心 0.50-0.65）


---
## 清理：Undeploy Index & 刪除 Endpoint
**Step 2 全部完成後才執行**，避免 e2-standard-16 持續計費（約 $0.64/小時）。

In [13]:
# ── 清理 Endpoint（Step 2 全部完成後才執行）────────────────
# e2-standard-16 約 $0.64/小時，不用時務必 undeploy

import configparser
import google.cloud.aiplatform as aip
from google.oauth2 import service_account

_cfg = configparser.ConfigParser()
_cfg.read(r"D:\yujui\GoogleCloud.ini", encoding="utf-8")

# GoogleCloud.ini 的 key 名稱帶有單引號，需用 strip 處理
def _get(section, key):
    sec = _cfg[section]
    # 先嘗試直接存取，再嘗試帶引號版本
    for k in (key, f"'{key}'"):
        if k in sec:
            return sec[k].strip("'\" ")
    raise KeyError(f'{key} not found in [{section}]')

_key_path   = _get("gcp", "service_account_key")
_project_id = _get("gcp", "project_id")
print(f"key_path   : {_key_path}")
print(f"project_id : {_project_id}")

_credentials = service_account.Credentials.from_service_account_file(
    _key_path,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
aip.init(project=_project_id, location="asia-east1", credentials=_credentials)

ENDPOINT_RESOURCE = "projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736"
DEPLOYED_INDEX_ID = "l1l7_seed_deployed"
index_endpoint = aip.MatchingEngineIndexEndpoint(ENDPOINT_RESOURCE)
print(f"Endpoint 載入：{index_endpoint.resource_name}")

print("Step 1：Undeploy Index...")
index_endpoint.undeploy_index(deployed_index_id=DEPLOYED_INDEX_ID)
print("  Undeploy 完成")

print("Step 2：刪除 Endpoint...")
index_endpoint.delete(force=True)
print("  Endpoint 刪除完成")
print("清理完成，e2-standard-16 計費已停止")

Step 1：Undeploy Index...
Undeploying index MatchingEngineIndexEndpoint index_endpoint: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736
Undeploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736/operations/7273946991778856960
MatchingEngineIndexEndpoint index_endpoint Undeployed index. Resource name: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736
  Undeploy 完成
Step 2：刪除 Endpoint...
Deleting MatchingEngineIndexEndpoint : projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736
MatchingEngineIndexEndpoint deleted. . Resource name: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736
Deleting MatchingEngineIndexEndpoint resource: projects/648642612568/locations/asia-east1/indexEndpoints/6952581458334580736
Delete MatchingEngineIndexEndpoint backing LRO: projects/648642612568/locations/asia-east1/